
# Regresión Lineal Simple - Clase 1

**Autora original:** Cecilia Oliva  
**Fecha de conversión a Python:** 17/03/2026

Este notebook es una adaptación de un archivo **R Markdown** a **Python**.  
Se mantuvo la estructura conceptual original, pero se reemplazó el código en R por alternativas en Python usando principalmente:

- `pandas` para lectura y manipulación de datos
- `matplotlib` y `seaborn` para gráficos
- `scipy.stats` para tests estadísticos
- `statsmodels` para modelos lineales y diagnóstico

> **Importante:** este notebook asume que los archivos `IMCinfantil.xlsx` y `gorriones.xlsx` están en la misma carpeta que este `.ipynb`.


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## Correlación e introducción al modelo lineal

## Ejemplo 1: Índice de masa corporal infantil


Realizamos los diagramas de dispersión de las variables del dataset **IMCinfantil.xlsx**.


In [ ]:

IMCinfantil = pd.read_excel("IMCinfantil.xlsx")

print(IMCinfantil.shape)
display(IMCinfantil.head())

base_ninios = IMCinfantil[["EDAD", "PESO", "TALLA", "IMC", "CC"]].copy()
base_ninios.head()


In [ ]:

# Matriz de dispersión
pd.plotting.scatter_matrix(base_ninios, figsize=(12, 10), diagonal='hist')
plt.suptitle("Diagramas de dispersión por pares", y=1.02)
plt.show()


In [ ]:

# Diagrama de dispersión sencillo
plt.figure(figsize=(7, 5))
plt.scatter(IMCinfantil["PESO"], IMCinfantil["CC"], color="black")
plt.xlabel("Peso")
plt.ylabel("Circunferencia de la Cintura")
plt.title("Peso vs Circunferencia de la Cintura")
plt.show()

# Diagrama con seaborn
plt.figure(figsize=(7, 5))
sns.scatterplot(data=IMCinfantil, x="PESO", y="CC", color="darkred")
plt.title("Peso vs Circunferencia de la Cintura")
plt.show()


### Boxplots y medidas resumen

In [ ]:

# Boxplots básicos
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=IMCinfantil["PESO"], ax=axes[0], color="lightgray")
axes[0].set_title("Boxplot vertical de PESO")

sns.boxplot(x=IMCinfantil["PESO"], ax=axes[1], color="lightgray")
axes[1].set_title("Boxplot horizontal de PESO")

fig.suptitle("Gráficos de cajas básicos", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:

IMCinf = IMCinfantil.copy()

# Cambio de etiquetas para CatPeso
mapa_peso = {
    "D": "Deficiente",
    "N": "Normal",
    "OB": "Obeso",
    "SO": "Sobrepeso"
}

IMCinf["CatPeso"] = IMCinf["CatPeso"].replace(mapa_peso)
orden_cat = ["Deficiente", "Normal", "Sobrepeso", "Obeso"]
IMCinf["CatPeso"] = pd.Categorical(IMCinf["CatPeso"], categories=orden_cat, ordered=True)

plt.figure(figsize=(8, 5))
sns.boxplot(data=IMCinf, x="CatPeso", y="CC", palette="terrain")
plt.title("Circunferencia de cintura según peso")
plt.xlabel("Categoría Peso")
plt.ylabel("Circunferencia de la Cintura")
plt.show()


In [ ]:

# Medidas resumen para PESO
print(IMCinfantil["PESO"].describe())
print("
Media:", IMCinfantil["PESO"].mean())
print("Mediana:", IMCinfantil["PESO"].median())
print("Desvío estándar:", IMCinfantil["PESO"].std())
print("Cuantiles 25%, 50%, 75%:")
print(IMCinfantil["PESO"].quantile([0.25, 0.50, 0.75]))


### Análisis de normalidad de las variables - Test de Shapiro Wilk

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].hist(IMCinfantil["PESO"], bins=10, edgecolor="darkred")
axes[0, 0].set_title("Histograma de Peso")
axes[0, 0].set_xlabel("Peso")

axes[0, 1].hist(IMCinfantil["CC"], bins=10, edgecolor="blue")
axes[0, 1].set_title("Histograma de Circ. Cintura")
axes[0, 1].set_xlabel("Circ. Cintura")

stats.probplot(IMCinfantil["PESO"], dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("QQ-plot de Peso")

stats.probplot(IMCinfantil["CC"], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("QQ-plot de Circ. Cintura")

plt.tight_layout()
plt.show()

print("Shapiro-Wilk para PESO")
print(stats.shapiro(IMCinfantil["PESO"]))
print("
Shapiro-Wilk para CC")
print(stats.shapiro(IMCinfantil["CC"]))


En ambos casos, si el valor-p es pequeño, se rechaza normalidad.

### Análisis de normalidad multivariada


En el archivo original se utilizaba el **test de Henze-Zirkler**.  
En Python, ese test no siempre está disponible por defecto en entornos básicos. Por eso, dejamos un bloque **opcional** que lo ejecuta si la librería `pingouin` está instalada. Si no lo está, el notebook sigue funcionando y muestra un mensaje aclaratorio.


In [ ]:

peso_cc = IMCinfantil[["PESO", "CC"]].dropna().copy()

try:
    from pingouin import multivariate_normality
    hz_result = multivariate_normality(peso_cc, alpha=0.05)
    print(hz_result)
except ImportError:
    print(
        "No se pudo correr Henze-Zirkler porque la librería 'pingouin' no está instalada.
"
        "Si querés usarlo, instalá primero: pip install pingouin"
    )


### Cálculo de correlación


Se calcula la correlación de **Pearson** entre las variables peso y circunferencia de la cintura.


In [ ]:

print("Correlación de Pearson:", IMCinfantil["PESO"].corr(IMCinfantil["CC"], method="pearson"))
print(stats.pearsonr(IMCinfantil["PESO"], IMCinfantil["CC"]))



Como puede no cumplirse el supuesto de normalidad bivariada, también calculamos la correlación de **Spearman**.


In [ ]:

print("Correlación de Spearman:", IMCinfantil["PESO"].corr(IMCinfantil["CC"], method="spearman"))
print(stats.spearmanr(IMCinfantil["PESO"], IMCinfantil["CC"]))


### Correlograma

In [ ]:

M = base_ninios.corr()
print("Matriz de correlación:")
display(M)

print("
Matriz de varianzas y covarianzas:")
display(base_ninios.cov())


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.heatmap(M, annot=False, cmap="coolwarm", square=True, ax=axes[0, 0])
axes[0, 0].set_title("Correlograma")

sns.heatmap(M, annot=True, fmt=".2f", cmap="coolwarm", square=True, ax=axes[0, 1])
axes[0, 1].set_title("Correlograma con valores")

sns.heatmap(M, mask=np.triu(np.ones_like(M, dtype=bool)), annot=True, fmt=".2f", cmap="coolwarm", square=True, ax=axes[1, 0])
axes[1, 0].set_title("Triángulo inferior")

sns.heatmap(M, mask=np.tril(np.ones_like(M, dtype=bool)), annot=True, fmt=".2f", cmap="coolwarm", square=True, ax=axes[1, 1])
axes[1, 1].set_title("Triángulo superior")

plt.tight_layout()
plt.show()


## Ejemplo 2: Gorriones

Realizamos los diagramas de dispersión de las variables del dataset **gorriones.xlsx**.

In [ ]:

gorr = pd.read_excel("gorriones.xlsx")
gorr = pd.DataFrame(gorr)

print(gorr.shape)
print(gorr.columns.tolist())
display(gorr.head())


In [ ]:

pd.plotting.scatter_matrix(gorr, figsize=(12, 10), diagonal='hist')
plt.suptitle("Diagramas de dispersión por pares - Gorriones", y=1.02)
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=gorr, x="Cabeza", y="Alas", color="darkred")
plt.title("Cabeza vs Alas")
plt.show()


### Análisis de normalidad de las variables - Test de Shapiro Wilk

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].hist(gorr["Cabeza"], bins=10, edgecolor="darkred")
axes[0, 0].set_title("Histograma de Cabeza")
axes[0, 0].set_xlabel("Cabeza")

axes[0, 1].hist(gorr["Alas"], bins=10, edgecolor="blue")
axes[0, 1].set_title("Histograma de Alas")
axes[0, 1].set_xlabel("Alas")

stats.probplot(gorr["Cabeza"], dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("QQ-plot de Cabeza")

stats.probplot(gorr["Alas"], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("QQ-plot de Alas")

plt.tight_layout()
plt.show()

print("Shapiro-Wilk para Cabeza")
print(stats.shapiro(gorr["Cabeza"]))
print("
Shapiro-Wilk para Alas")
print(stats.shapiro(gorr["Alas"]))


### Análisis de normalidad multivariada

In [ ]:

cabeza_alas = gorr[["Cabeza", "Alas"]].dropna().copy()

try:
    from pingouin import multivariate_normality
    hz_result_gorr = multivariate_normality(cabeza_alas, alpha=0.05)
    print(hz_result_gorr)
except ImportError:
    print(
        "No se pudo correr Henze-Zirkler porque la librería 'pingouin' no está instalada.
"
        "Si querés usarlo, instalá primero: pip install pingouin"
    )


### Cálculo de correlación

Se calcula la correlación de Pearson entre la longitud de la cabeza y la longitud de las alas.

In [ ]:

print("Correlación de Pearson:", gorr["Cabeza"].corr(gorr["Alas"], method="pearson"))
print(stats.pearsonr(gorr["Cabeza"], gorr["Alas"]))


## Modelos lineales

Supongamos que queremos estudiar la longitud del largo de los gorriones en función de las otras variables a través de un modelo lineal.

In [ ]:

model_gorr1 = smf.ols("Largo ~ Alas", data=gorr).fit()
model_gorr2 = smf.ols("Largo ~ Cabeza", data=gorr).fit()
model_gorr3 = smf.ols("Largo ~ Pata", data=gorr).fit()
model_gorr4 = smf.ols("Largo ~ Cuerpo", data=gorr).fit()
model_gorr5 = smf.ols("Largo ~ Alas + Pata", data=gorr).fit()
model_gorr = smf.ols("Largo ~ Alas + Cabeza + Pata + Cuerpo", data=gorr).fit()

print(model_gorr1.summary())
print(model_gorr2.summary())
print(model_gorr3.summary())
print(model_gorr4.summary())
print(model_gorr5.summary())
print(model_gorr.summary())


## Comparación de modelos

### Usando ANOVA

In [ ]:

print(anova_lm(model_gorr1, model_gorr5))
print(anova_lm(model_gorr3, model_gorr5))
print(anova_lm(model_gorr2, model_gorr))
print(anova_lm(model_gorr1, model_gorr))
print(anova_lm(model_gorr3, model_gorr))
print(anova_lm(model_gorr5, model_gorr))


### Usando AIC y BIC

In [ ]:

comparacion_modelos = pd.DataFrame({
    "Modelo": ["model_gorr1", "model_gorr2", "model_gorr3", "model_gorr5", "model_gorr"],
    "AIC": [model_gorr1.aic, model_gorr2.aic, model_gorr3.aic, model_gorr5.aic, model_gorr.aic],
    "BIC": [model_gorr1.bic, model_gorr2.bic, model_gorr3.bic, model_gorr5.bic, model_gorr.bic]
})

comparacion_modelos


## Estimación de coeficientes y recta de regresión

### Tests tipo Wald


En `statsmodels`, los contrastes tipo Wald se pueden hacer con `wald_test`.  
Los nombres de coeficientes incluyen el intercepto como `Intercept`.


In [ ]:

# Comparación conjunta de Cabeza y Pata en el modelo múltiple
print(model_gorr.wald_test("Cabeza = 0, Pata = 0"))

# Comparación conjunta de Alas y Cabeza en el modelo múltiple
print(model_gorr.wald_test("Alas = 0, Cabeza = 0"))

# Tests individuales
print(model_gorr.wald_test("Alas = 0"))
print(model_gorr1.wald_test("Alas = 0"))
print(model_gorr.wald_test("Cabeza = 0"))
print(model_gorr2.wald_test("Cabeza = 0"))
print(model_gorr.wald_test("Pata = 0"))
print(model_gorr3.wald_test("Pata = 0"))
print(model_gorr.wald_test("Cuerpo = 0"))
print(model_gorr4.wald_test("Cuerpo = 0"))



**Observación:** cuando una variable resulta significativa en un modelo simple pero no en un modelo múltiple, una explicación frecuente es la presencia de colinealidad o solapamiento de información entre predictores.


### Intervalos de confianza

In [ ]:
model_gorr.conf_int()

### Gráfico de recta de regresión

In [ ]:

plt.figure(figsize=(7, 5))
sns.regplot(data=gorr, x="Alas", y="Largo", scatter_kws={"color": "firebrick", "s": 40}, line_kws={"color": "black"}, ci=None)
plt.title("Largo ~ Alas")
plt.xlabel("Alas")
plt.ylabel("Largo")
plt.show()


## Predicción e intervalos de confianza

Genero nuevos datos en el rango de X.

In [ ]:
nuevosdatos = np.linspace(gorr["Alas"].min(), gorr["Alas"].max(), 100)
nuevosdatos[:5]

Predigo valores de `Largo` para nuevos valores de `Alas`.

In [ ]:

new_df = pd.DataFrame({"Alas": nuevosdatos})
predicciones = model_gorr1.predict(new_df)
predicciones.head()


Se puede predecir por fuera del rango observado, aunque la confiabilidad puede ser menor.

In [ ]:
model_gorr1.predict(pd.DataFrame({"Alas": [280]}))

Predigo `Y` y agrego su intervalo de confianza para cada uno de los nuevos datos generados.

In [ ]:

pred_summary = model_gorr1.get_prediction(new_df).summary_frame(alpha=0.05)
pred_summary.head()


Armo el gráfico con la recta ajustada y las curvas de intervalos de confianza.

In [ ]:

plt.figure(figsize=(8, 5))
plt.scatter(gorr["Alas"], gorr["Largo"], color="firebrick", s=35, label="Datos")
plt.plot(new_df["Alas"], pred_summary["mean"], color="black", label="Recta ajustada")
plt.plot(new_df["Alas"], pred_summary["mean_ci_lower"], linestyle="--", color="red", label="IC inferior")
plt.plot(new_df["Alas"], pred_summary["mean_ci_upper"], linestyle="--", color="green", label="IC superior")
plt.xlabel("Alas")
plt.ylabel("Largo")
plt.title("Largo vs Alas")
plt.legend()
plt.show()


In [ ]:

# Alternativa con seaborn, mostrando banda de confianza al 95%
plt.figure(figsize=(8, 5))
sns.regplot(data=gorr, x="Alas", y="Largo", scatter_kws={"color": "firebrick", "s": 40}, line_kws={"color": "black"}, ci=95)
plt.title("Largo vs Alas")
plt.show()


## Validación de supuestos y análisis diagnóstico

### Análisis de normalidad de los residuos - Test de Shapiro Wilk

In [ ]:

gorr2 = gorr.copy()
gorr2["prediccion"] = model_gorr1.fittedvalues
gorr2["residuos"] = model_gorr1.resid

display(gorr2.head())


In [ ]:

plt.figure(figsize=(7, 5))
sns.histplot(gorr2["residuos"], kde=True)
plt.title("Histograma de los residuos")
plt.show()

fig = plt.figure(figsize=(6, 5))
stats.probplot(model_gorr1.resid, dist="norm", plot=plt)
plt.title("QQ-plot de residuos")
plt.show()

print(stats.shapiro(model_gorr1.resid))


### Gráfico de análisis de homocedasticidad

Si se observa algún patrón visual claro, podría cuestionarse el supuesto de homocedasticidad.

In [ ]:

plt.figure(figsize=(8, 5))
sns.scatterplot(data=gorr2, x="prediccion", y="residuos", hue="residuos", palette="coolwarm", legend=False)
for _, row in gorr2.iterrows():
    plt.plot([row["prediccion"], row["prediccion"]], [0, row["residuos"]], color="gray", alpha=0.2)
plt.axhline(0, color="black")
plt.title("Distribución de los residuos")
plt.xlabel("Predicción modelo")
plt.ylabel("Residuo")
plt.show()


### Test de Breusch-Pagan

Analizamos si los residuos tienen varianza constante (homocedasticidad).

In [ ]:

bp_test = het_breuschpagan(model_gorr1.resid, model_gorr1.model.exog)
bp_labels = ["LM Statistic", "LM-Test p-value", "F-Statistic", "F-Test p-value"]
print(dict(zip(bp_labels, bp_test)))


Si el valor-p no es pequeño, no se rechaza homocedasticidad.

### Gráfico de análisis de independencia de las observaciones

Analizamos los residuos según el orden de las observaciones para ver si se observa dependencia.

In [ ]:

plt.figure(figsize=(8, 5))
plt.scatter(np.arange(1, len(gorr2) + 1), gorr2["residuos"], c=gorr2["residuos"], cmap="coolwarm")
plt.plot(np.arange(1, len(gorr2) + 1), gorr2["residuos"], linewidth=0.7)
plt.axhline(0, color="black")
plt.title("Distribución de los residuos")
plt.xlabel("Índice")
plt.ylabel("Residuo")
plt.show()


No se detecta un patrón si los puntos no muestran tendencia ni estructura clara.

### Test de Durbin-Watson

Analizamos autocorrelación de los residuos. Valores cercanos a 2 suelen indicar ausencia de autocorrelación.

In [ ]:
print("Durbin-Watson:", durbin_watson(model_gorr1.resid))